In [1]:
import pandas as pd
import numpy as np

# Cargamos los datos crudos desde tu carpeta data/raw
ruta_archivo = '../data/raw/historical_results.csv'
df = pd.read_csv(ruta_archivo)

# Vemos la forma básica del dataset y las primeras 5 filas
print(f"El dataset tiene {df.shape[0]} filas y {df.shape[1]} columnas.")
df.head()

El dataset tiene 25268 filas y 9 columnas.


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,2000-01-04,Egypt,Togo,2.0,1.0,Friendly,Aswan,Egypt,False
1,2000-01-07,Tunisia,Togo,7.0,0.0,Friendly,Tunis,Tunisia,False
2,2000-01-08,Trinidad and Tobago,Canada,0.0,0.0,Friendly,Port of Spain,Trinidad and Tobago,False
3,2000-01-09,Burkina Faso,Gabon,1.0,1.0,Friendly,Ouagadougou,Burkina Faso,False
4,2000-01-09,Guatemala,Armenia,1.0,1.0,Friendly,Los Angeles,United States,True


In [2]:
# Revisamos los tipos de datos de cada columna
print("--- Información del Dataset ---")
df.info()

print("\n--- Valores Nulos por Columna ---")
# Contamos cuántos valores vacíos hay en cada columna
print(df.isnull().sum())

--- Información del Dataset ---
<class 'pandas.DataFrame'>
RangeIndex: 25268 entries, 0 to 25267
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        25268 non-null  str    
 1   home_team   25268 non-null  str    
 2   away_team   25268 non-null  str    
 3   home_score  25196 non-null  float64
 4   away_score  25196 non-null  float64
 5   tournament  25268 non-null  str    
 6   city        25268 non-null  str    
 7   country     25268 non-null  str    
 8   neutral     25268 non-null  bool   
dtypes: bool(1), float64(2), str(6)
memory usage: 1.6 MB

--- Valores Nulos por Columna ---
date           0
home_team      0
away_team      0
home_score    72
away_score    72
tournament     0
city           0
country        0
neutral        0
dtype: int64


In [3]:
# Eliminamos las filas que tengan valores nulos específicamente en los goles
df = df.dropna(subset=['home_score', 'away_score'])

# Comprobamos cómo quedó nuestro dataset
print(f"El dataset limpio ahora tiene {len(df)} partidos válidos.")
print(df.isnull().sum())

El dataset limpio ahora tiene 25196 partidos válidos.
date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64


In [4]:
import numpy as np

# 1. Definimos las reglas lógicas del fútbol
condiciones = [
    df['home_score'] > df['away_score'],  # Victoria Local
    df['home_score'] < df['away_score']   # Victoria Visitante
]

# 2. Las etiquetas que usaremos en los gráficos
etiquetas = ['HW', 'AW']

# 3. Aplicamos la lógica: si no gana el local ni el visitante, por defecto es Empate ('D')
df['target'] = np.select(condiciones, etiquetas, default='D')

# 4. Creamos la versión factorizada para la matriz de correlación y el modelo (2=Gana Local, 1=Empate, 0=Pierde)
df['target_numeric'] = df['target'].map({'HW': 2, 'D': 1, 'AW': 0})

# 5. Verificamos cómo quedó el balance de clases
print("--- Porcentaje histórico de resultados ---")
print(df['target'].value_counts(normalize=True) * 100)

df[['home_team', 'away_team', 'home_score', 'away_score', 'target', 'target_numeric']].head()

--- Porcentaje histórico de resultados ---
target
HW    48.098905
AW    28.595809
D     23.305287
Name: proportion, dtype: float64


,home_team,away_team,home_score,away_score,target,target_numeric
0,Egypt,Togo,2.0,1.0,HW,2
1,Tunisia,Togo,7.0,0.0,HW,2
2,Trinidad and Tobago,Canada,0.0,0.0,D,1
3,Burkina Faso,Gabon,1.0,1.0,D,1
4,Guatemala,Armenia,1.0,1.0,D,1


In [5]:
# 1. Transformamos la columna booleana 'neutral' (True/False) a formato binario (1/0)
df['neutral_numeric'] = df['neutral'].astype(int)

# 2. Eliminamos las columnas de texto geográfico que solo aportan ruido
df = df.drop(columns=['city', 'country'])

# 3. Filtramos los partidos con resultados extremos (más de 10 goles)
tamaño_previo = len(df)
df = df[(df['home_score'] <= 10) & (df['away_score'] <= 10)]
partidos_eliminados = tamaño_previo - len(df)

print(f"Limpieza exitosa. Se eliminaron {partidos_eliminados} partidos atípicos históricos.")
print("\n--- Columnas Finales Listas para Modelar ---")
print(list(df.columns))

Limpieza exitosa. Se eliminaron 93 partidos atípicos históricos.

--- Columnas Finales Listas para Modelar ---
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'neutral', 'target', 'target_numeric', 'neutral_numeric']


In [6]:
import os

# 1. Creamos una carpeta para los datos procesados/intermedios si no existe
os.makedirs('../data/interim', exist_ok=True)

# 2. Guardamos el dataset limpio
df.to_csv('../data/interim/partidos_limpios.csv', index=False)

print("¡Punto de control guardado! Dataset listo en 'data/interim/partidos_limpios.csv'")

¡Punto de control guardado! Dataset listo en 'data/interim/partidos_limpios.csv'
